# Testing
### Bevölkerung und Anzahl Nationalitäten nach Stadtquartier, seit 1993


Anzahl unterschiedliche Staatsangehörigkeiten und wirtschaftliche Wohnbevölkerung der Stadt Zürich nach Staatsangehörigkeit, Statistischem Stadtquartier und Jahr, seit 1993.

Datum: 15.03.2023


### Importiere die notwendigen Packages

In [117]:
#%pip install geopandas altair fiona requests folium mplleaflet contextily seaborn datetime plotly leafmap

In [118]:
import altair as alt
import datetime
import folium 
import geopandas as gpd
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
#import pivottablejs
#from pivottablejs import pivot_ui
import plotly.express as px
import requests
import seaborn as sns


In [119]:
SSL_VERIFY = False
# evtl. SSL_VERIFY auf False setzen wenn die Verbindung zu https://www.gemeinderat-zuerich.ch nicht klappt (z.B. wegen Proxy)
# Um die SSL Verifikation auszustellen, bitte die nächste Zeile einkommentieren ("#" entfernen)
# SSL_VERIFY = False

In [120]:
if not SSL_VERIFY:
    import urllib3
    urllib3.disable_warnings()

Definiere Settings. Hier das Zahlenformat von Float-Werten (z.B. *'{:,.2f}'.format* mit Komma als Tausenderzeichen), 

In [121]:
#pd.options.display.float_format = lambda x : '{:,.1f}'.format(x) if (np.isnan(x) | np.isinf(x)) else '{:,.0f}'.format(x) if int(x) == x else '{:,.1f}'.format(x)
pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.width', 100)
pd.set_option('display.max_columns', 15)

### Zeitvariabeln
Bestimme den aktuellst geladenen Monat. Hier ist es der Stand vor 2 Monaten. 
Bestimme noch weitere evt. sinnvolle Zeitvariabeln.

Zum Unterschied zwischen import datetime und from datedtime import datetime, siehe https://stackoverflow.com/questions/15707532/import-datetime-v-s-from-datetime-import-datetime

Zuerst die Zeitvariabeln als Strings

In [122]:
#today_date = datetime.date.today()
#date_time = datetime.datetime.strptime(date_time_string, '%Y-%m-%d %H:%M')
now = datetime.date.today()
date_today = now.strftime("%Y-%m-%d")
year_today = now.strftime("%Y")
month_today = now.strftime("%m")
day_today = now.strftime("%d")


Und hier noch die Zeitvariabeln als Integers:
- `aktuellesJahr`
- `aktuellerMonat`: Der gerade jetzt aktuelle Monat
- `selectedMonat`: Der aktuellste Monat in den Daten. In der Regel zwei Monate her.

In [123]:
#now = datetime.now() 
int_times = now.timetuple()

aktuellesJahr = int_times[0]
aktuellerMonat = int_times[1]
selectedMonat = int_times[1]-2

print(aktuellesJahr, 
      aktuellerMonat,
    'datenstand: ', selectedMonat,
     int_times)


2024 3 datenstand:  1 time.struct_time(tm_year=2024, tm_mon=3, tm_mday=10, tm_hour=0, tm_min=0, tm_sec=0, tm_wday=6, tm_yday=70, tm_isdst=-1)


Berechne die Variable Epoche um später das SAS-Datum in ein Unix-Datum umzuwandeln. Bei SAS beginnt die Epoche am 1.1.1960. Bei Unix am 1.1.1970.
Diese Variable wird beim CSV-Import benötigt.

In [124]:
epoch = datetime.datetime(1960, 1, 1)

### Setze einige Pfadvariabeln

- Der Packagename ist eigentlich der **Verzeichnisname** unter dem die Daten und Metadaten auf der Dropzone abgelegt werden.
- Definiert wird er bei SASA-Prozessen auf dem **Produkte-Sharepoint ([Link](https://kollaboration.intranet.stzh.ch/orga/ssz-produkte/Lists/SASA_Outputs/PersonalViews.aspx?PageView=Personal&ShowWebPart={6087A3E7-8AC8-40BA-8278-DECFACE124FF}))**.
- Der Packagename wird auf CKAN teil der URL, daher ist die exakte Schreibweise wichtig.

Beachte: im Packagename müssen alle Buchstaben **klein** geschrieben werden. Dies weil CKAN aus grossen kleine Buchstaben macht.

**BITTE HIER ANPASSEN**

In [125]:
package_name = "bev_bestand_jahr_anz_nationen_quartier_od3363"

In [126]:
dataset_name = "BEV336OD3363.csv"

**Statische Pfade in DWH-Dropzones**

In [127]:
dropzone_path_integ = r"\\szh\ssz\applikationen\OGD_Dropzone\INT_DWH"

In [128]:
dropzone_path_prod = r"\\szh\ssz\applikationen\OGD_Dropzone\DWH"

**Statische Pfade CKAN-URLs**

In [129]:
ckan_integ_url ="https://data.integ.stadt-zuerich.ch/dataset/int_dwh_"

In [130]:
ckan_prod_url ="https://data.stadt-zuerich.ch/dataset/"

### Checke die Metadaten auf der CKAN INTEG- oder PROD-Webseite

Offenbar lassen sich aktuell im Markdownteil keine Variabeln ausführen, daher gehen wir wie unten gezeigt vor. Siehe dazu: https://data-dive.com/jupyterlab-markdown-cells-include-variables
Instead of setting the cell to Markdown, create Markdown from withnin a code cell! We can just use python variable replacement syntax to make the text dynamic

In [131]:
from IPython.display import Markdown as md

In [132]:
md(" **1. Dataset auf INTEG-Datakatalog:** Link {} ".format(ckan_integ_url+package_name))

 **1. Dataset auf INTEG-Datakatalog:** Link https://data.integ.stadt-zuerich.ch/dataset/int_dwh_bev_bestand_jahr_anz_nationen_quartier_od3363 

In [133]:
md(" **2. Dataset auf PROD-Datakatalog:** Link {} ".format(ckan_prod_url+package_name))

 **2. Dataset auf PROD-Datakatalog:** Link https://data.stadt-zuerich.ch/dataset/bev_bestand_jahr_anz_nationen_quartier_od3363 

### Importiere einen Datensatz 

Definiere zuerst folgende Werte:
1) Kommt der Datensatz von PROD oder INTEG?
2) Beziehst Du den Datensatz direkt ab der DROPZONE oder aus dem INTERNET?

In [134]:
#Die Datasets sind nur zum Testen auf INT-DWH-Dropzone. Wenn der Test vorbei ist, sind sie auf PROD. 
# Über den Status kann man einfach switchen

status = "int"; #prod vs something else
data_source = "web"; #dropzone vs something else
print(status+" - "+ data_source)

int - web


In [135]:
# Filepath
if status == "prod":
    if data_source == "dropzone":
            fp = dropzone_path_prod+"\\"+ package_name +"\\"+dataset_name
            print("fp lautet:"+fp)
    else:
        #fp = r"https://data.stadt-zuerich.ch/dataset/bau_neubau_whg_bausm_rinh_geb_projstatus_quartier_seit2009_od5011/download/BAU501OD5011.csv"
        fp = ckan_prod_url+package_name+'/download/'+dataset_name
        print("fp lautet:"+fp)
else:
    if data_source == "dropzone":
        fp = dropzone_path_integ+"\\"+ package_name +"\\"+dataset_name
        print("fp lautet:"+fp)
    else:
        #fp = r"https://data.stadt-zuerich.ch/dataset/bau_neubau_whg_bausm_rinh_geb_projstatus_quartier_seit2009_od5011/download/BAU501OD5011.csv"
        fp = ckan_integ_url+package_name+'/download/'+dataset_name
        print("fp lautet:"+fp)


fp lautet:https://data.integ.stadt-zuerich.ch/dataset/int_dwh_bev_bestand_jahr_anz_nationen_quartier_od3363/download/BEV336OD3363.csv


Beachte, wie das SAS Datum (ohne Format) in ein UNIX Datum umgerechnet und als Datumsformat dargestellt wird! Siehe dazu `https://stackoverflow.com/questions/26923564/convert-sas-numeric-to-python-datetime`

In [136]:
# Read the data
if data_source == "dropzone":
    data2betested = pd.read_csv(
        fp
        , sep=','
        ,parse_dates=['StichtagDatJahr']
        ,low_memory=False
    )
    print("dropzone")
else:
    r = requests.get(fp, verify=False)  
    r.encoding = 'utf-8'
    data2betested = pd.read_csv(
        io.StringIO(r.text)
        ,parse_dates=['StichtagDatJahr']
        # KONVERTIERE DAS SAS DATUM IN EIN UNIXDATUM UND FORMATIERE ES
        #, date_parser=lambda s: epoch + datetime.timedelta(days=int(s))
        ,low_memory=False)
    print("web")

data2betested.dtypes

web


StichtagDatJahr    datetime64[ns]
QuarSort                    int64
QuarCd                      int64
QuarLang                   object
DatenstandCd               object
AnzBestWir                  int64
AnzNat                      int64
dtype: object

Berechne weitere Attribute falls notwendig

In [137]:
data2betested = (
    data2betested
    .copy()
    .assign(
        #Aktualisierungs_Datum_str= lambda x: x.Aktualisierungs_Datum.astype(str),
        Jahr = lambda x: x.StichtagDatJahr,
        Jahr_str = lambda x: x.StichtagDatJahr.astype(str),
        Jahr_nbr = lambda x: x.Jahr.dt.year,
    )
    .sort_values('StichtagDatJahr', ascending=False)
    )
data2betested.dtypes

StichtagDatJahr    datetime64[ns]
QuarSort                    int64
QuarCd                      int64
QuarLang                   object
DatenstandCd               object
AnzBestWir                  int64
AnzNat                      int64
Jahr               datetime64[ns]
Jahr_str                   object
Jahr_nbr                    int64
dtype: object

Minimales und maximales Jahr im Datensatz

In [138]:
data_max_date = str(max(data2betested.Jahr).year)
data_min_date = str(min(data2betested.Jahr).year)

print(f"Die Daten haben ein Minimumjahr von {data_min_date} und ein Maximumjahr von {data_max_date}")

Die Daten haben ein Minimumjahr von 1993 und ein Maximumjahr von 2023


### Einfache Datentests

 - 1) Zeige eine kurze Vorschau der importierten Daten
 - 2) Weise die Datentypen aus
 - 3) Zeige die Shape (Umfang) des Datensatzes an

In [139]:
#data2betested.head(6)

In [140]:
data2betested.dtypes

StichtagDatJahr    datetime64[ns]
QuarSort                    int64
QuarCd                      int64
QuarLang                   object
DatenstandCd               object
AnzBestWir                  int64
AnzNat                      int64
Jahr               datetime64[ns]
Jahr_str                   object
Jahr_nbr                    int64
dtype: object

In [141]:
data2betested.shape

(1054, 10)

Beschreibe einzelne Attribute

In [142]:
data2betested.describe()

,QuarSort,QuarCd,AnzBestWir,AnzNat,Jahr_nbr
count,1054,1054,1054,1054,1054
mean,65,65,11443,94,2008
std,36,36,7432,23,9
min,11,11,634,30,1993
25%,33,33,5548,81,2000
50%,67,67,10247,97,2008
75%,92,92,16626,111,2016
max,123,123,36315,137,2023


Wie viele Nullwerte gibt es im Datensatz?

In [143]:
data2betested.isnull().sum()

StichtagDatJahr    0
QuarSort           0
QuarCd             0
QuarLang           0
DatenstandCd       0
AnzBestWir         0
AnzNat             0
Jahr               0
Jahr_str           0
Jahr_nbr           0
dtype: int64

Welches sind die Quartiere ohne Werte bei AnzBestWir?

In [144]:
data2betested[np.isnan(data2betested.AnzBestWir)]

,StichtagDatJahr,QuarSort,QuarCd,QuarLang,DatenstandCd,AnzBestWir,AnzNat,Jahr,Jahr_str,Jahr_nbr


### Verwende das Datum als Index

While we did already parse the `datetime` column into the respective datetime type, it currently is just a regular column. 
**To enable quick and convenient queries and aggregations, we need to turn it into the index of the DataFrame**

In [145]:
data2betested = data2betested.set_index("StichtagDatJahr")
data2betested = data2betested.sort_index()

In [146]:
data2betested.info()
data2betested.index.year.unique()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1054 entries, 1993-01-01 to 2023-01-01
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   QuarSort      1054 non-null   int64         
 1   QuarCd        1054 non-null   int64         
 2   QuarLang      1054 non-null   object        
 3   DatenstandCd  1054 non-null   object        
 4   AnzBestWir    1054 non-null   int64         
 5   AnzNat        1054 non-null   int64         
 6   Jahr          1054 non-null   datetime64[ns]
 7   Jahr_str      1054 non-null   object        
 8   Jahr_nbr      1054 non-null   int64         
dtypes: datetime64[ns](1), int64(5), object(3)
memory usage: 82.3+ KB


Int64Index([1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006,
            2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020,
            2021, 2022, 2023],
           dtype='int64', name='StichtagDatJahr')

### Einfache Visualisierungen zur Plausi

Exploriere die Daten mit Pivottable.JS

In [147]:
#pivot_ui(data2betested)

### Zeitpunkte und Zeiträume abfragen

A particular powerful feature of the Pandas DataFrame is its indexing capability that also works using time-based entities, such as dates and times. We have already created the index above, so let's put it to use.

In [148]:
data2betested.loc[data_max_date].head(2)


,QuarSort,QuarCd,QuarLang,DatenstandCd,AnzBestWir,AnzNat,Jahr,Jahr_str,Jahr_nbr
StichtagDatJahr,,,,,,,,,
2023-01-01,12,12,Hochschulen,D,695,46,2023-01-01,2023-01-01,2023
2023-01-01,13,13,Lindenhof,D,1046,49,2023-01-01,2023-01-01,2023


### Visualisierungen nach Zeitausschnitten

In [149]:
data2betested.columns

Index(['QuarSort', 'QuarCd', 'QuarLang', 'DatenstandCd', 'AnzBestWir', 'AnzNat', 'Jahr',
       'Jahr_str', 'Jahr_nbr'],
      dtype='object')

#### Stadtquartiere nach Anzahl Nationalitäten

In [150]:
myTitle="Stadtquartiere nach Anzahl Nationalitäten, " +data_min_date + "- "+data_max_date

highlight = alt.selection(type='single', on='mouseover',
                          fields=['QuarLang'], nearest=True)
#x='date:StichtagDatJahr',
base = alt.Chart(data2betested.reset_index().query('AnzBestWir>10'), title=myTitle).encode(
    x=alt.X('StichtagDatJahr', axis=alt.Axis(title='Jahr'))# , axis=alt.Axis(format='%', title='percentage')
    , y=alt.X('AnzNat', axis=alt.Axis(title='Anz. Nationen'))
    , color=alt.Color('QuarLang', legend=alt.Legend(title="Altersgruppen", orient="right"))  
    ,tooltip=['StichtagDatJahr', 'QuarLang','AnzBestWir', 'AnzNat']    
)
points = base.mark_circle().encode(
    opacity=alt.value(0.75)
).add_selection(
    highlight
).properties(
    width=750 , height=350
)
lines = base.mark_line().encode(
    size=alt.condition(~highlight, alt.value(0.5), alt.value(4))
).interactive()

lines + points

alt.LayerChart(...)

In [152]:
myAgg = data2betested.loc[data_max_date]  \
    .groupby(['StichtagDatJahr', 'QuarLang','QuarSort']) \
    .agg(sum_WBev=('AnzBestWir', 'sum'), sum_AnzNat=('AnzNat', 'sum')) \
    .sort_values('sum_AnzNat', ascending=False) 

myAgg.reset_index()

,StichtagDatJahr,QuarLang,QuarSort,sum_WBev,sum_AnzNat
0,2023-01-01,Altstetten,92,36315,135
1,2023-01-01,Oerlikon,115,23959,133
2,2023-01-01,Seebach,119,27677,132
3,2023-01-01,Wollishofen,21,21181,125
4,2023-01-01,Unterstrass,61,24801,123
5,2023-01-01,Hirzenbach,123,13251,122
6,2023-01-01,Affoltern,111,27165,122
7,2023-01-01,Schwamendingen-Mitte,122,11421,120
8,2023-01-01,Höngg,101,24674,119
9,2023-01-01,Sihlfeld,34,21753,118


**Sharepoint als gecheckt markieren!**

Record auf Sharepoint: **[Link](http://kollaboration.intranet.stzh.ch/orga/ssz-produkte/Lists/SASA_Outputs/EditForm.aspx?ID=157&Source=%2Forga%2Fssz%2Dprodukte%2FLists%2FSASA%5FOutputs)**